# Phase-2 Step 2: Multi-Dataset Harmonization

**Objective:** Standardize data across NASA batteries for consistent training/testing.

**Tasks:**
1. Load all audited batteries (B0005, B0006, B0007)
2. Resample to common time base
3. Extract consistent physics features
4. Normalize features
5. Create train/test split
6. Save harmonized data

**Output:**
- Harmonized feature DataFrame
- Normalization parameters
- Train/test splits ready for modeling

In [ ]:
# --- Portable setup (Windows / macOS / Linux) — do not hardcode paths ---
import importlib.util
from pathlib import Path

def _load_notebook_setup():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "notebooks" / "notebook_setup.py",
        cwd / "notebook_setup.py",
    ]
    if cwd.name == "notebooks":
        candidates.insert(0, cwd / "notebook_setup.py")
    for fp in candidates:
        if fp.is_file():
            spec = importlib.util.spec_from_file_location("_notebook_setup", fp)
            mod = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            return mod
    raise FileNotFoundError(
        "Missing notebooks/notebook_setup.py. Unzip the full project, open a terminal, "
        "cd into the repo root, then run: jupyter notebook"
    )

_ns = _load_notebook_setup()
PROJECT_ROOT = _ns.setup()
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


import sys
import os

# ============================================================================
# SETUP
# ============================================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

from data_harmonization import DataHarmonizer, run_harmonization
from utils import plot_style_setup

plot_style_setup()

# Load config
with open('phase2_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("✓ Imports complete")
print(f"✓ Configuration loaded")

## 1. Initialize Harmonizer

In [ ]:
# Create harmonizer
harmonizer = DataHarmonizer(config, PROJECT_ROOT)

print(f"Output directory: {harmonizer.output_dir}")
print(f"Timestamp: {harmonizer.timestamp}")

## 2. Load and Harmonize All Batteries

In [ ]:
# Get battery configurations
battery_configs = [
    config['datasets']['nasa_train'],     # B0005
    config['datasets']['nasa_test_1'],    # B0006
    config['datasets']['nasa_test_2']     # B0007
]

print("Batteries to process:")
for bc in battery_configs:
    print(f"  - {bc['battery_id']}: {bc['path']}")

In [ ]:
# Load and harmonize
batteries = harmonizer.load_and_harmonize_nasa(battery_configs)

## 3. Visualize Harmonized Data

In [ ]:
# Plot SOH degradation for all batteries
fig, ax = plt.subplots(figsize=(12, 6))

colors = {'B0005': 'blue', 'B0006': 'orange', 'B0007': 'green'}

for battery_id, cycles in batteries.items():
    cycle_nums = [c.cycle_number for c in cycles]
    soh_values = [c.soh for c in cycles]
    
    ax.plot(cycle_nums, soh_values, 'o-', 
            label=f'{battery_id} (n={len(cycles)})', 
            color=colors.get(battery_id, 'gray'),
            markersize=3, alpha=0.7)

ax.axhline(y=80, color='r', linestyle='--', alpha=0.5, label='EOL Threshold (80%)')
ax.set_xlabel('Cycle Number', fontsize=12)
ax.set_ylabel('SOH (%)', fontsize=12)
ax.set_title('SOH Degradation: All Harmonized Batteries', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/phase2/figures/soh_all_batteries.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: results/phase2/figures/soh_all_batteries.png")

In [ ]:
# Plot data quality scores
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, (battery_id, cycles) in enumerate(batteries.items()):
    quality_scores = [c.data_quality_score for c in cycles]
    
    axes[i].hist(quality_scores, bins=20, edgecolor='black', alpha=0.7, color=colors.get(battery_id, 'gray'))
    axes[i].axvline(x=np.mean(quality_scores), color='r', linestyle='--', 
                    label=f'Mean: {np.mean(quality_scores):.2f}')
    axes[i].set_xlabel('Quality Score')
    axes[i].set_ylabel('Count')
    axes[i].set_title(f'{battery_id}: Data Quality Distribution')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Data Quality Scores by Battery', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/phase2/figures/data_quality_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: results/phase2/figures/data_quality_distribution.png")

## 4. Extract Physics-Guided Features

In [ ]:
# Extract features from all cycles
features_df = harmonizer.extract_features()

print(f"\nFeature DataFrame shape: {features_df.shape}")
print(f"\nFeatures extracted:")
for col in features_df.columns:
    print(f"  - {col}")

In [ ]:
# Show sample of features
print("\nSample features (first 5 rows):")
features_df.head()

In [ ]:
# Feature statistics by battery
print("\nFeature Statistics by Battery:")
features_df.groupby('battery_id')[['soh', 'capacity', 'ica_peak_voltage', 'dcir', 'temp_delta']].agg(['mean', 'std']).round(4)

## 5. Feature Correlation Analysis

In [ ]:
# Correlation with SOH
numeric_cols = features_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['cycle_number', 'original_index', 'has_spikes']]

correlations = features_df[numeric_cols].corr()['soh'].drop('soh').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
correlations.plot(kind='barh', ax=ax, color=['green' if x > 0 else 'red' for x in correlations])
ax.set_xlabel('Correlation with SOH')
ax.set_title('Feature Correlations with SOH', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('results/phase2/figures/feature_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop positive correlations:")
print(correlations.head(5))
print("\nTop negative correlations:")
print(correlations.tail(5))

## 6. Normalize Features

In [ ]:
# Normalize using z-score
features_normalized = harmonizer.normalize_features(method='z-score')

print(f"\nNormalized features shape: {features_normalized.shape}")
print(f"\nNormalization parameters saved: {len(harmonizer.normalization_params)} features")

## 7. Create Train/Test Split

In [ ]:
# Split by battery
train_df, test_df = harmonizer.create_train_test_split(
    train_battery='B0005',
    test_batteries=['B0006', 'B0007']
)

print(f"\nTrain set (B0005):")
print(f"  Samples: {len(train_df)}")
print(f"  SOH range: [{train_df['soh'].min():.2f}, {train_df['soh'].max():.2f}]%")

print(f"\nTest set (B0006 + B0007):")
print(f"  Samples: {len(test_df)}")
print(f"  SOH range: [{test_df['soh'].min():.2f}, {test_df['soh'].max():.2f}]%")

In [ ]:
# Visualize train/test distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# SOH distribution
axes[0].hist(train_df['soh'], bins=30, alpha=0.7, label='Train (B0005)', color='blue', edgecolor='black')
axes[0].hist(test_df['soh'], bins=30, alpha=0.7, label='Test (B0006+B0007)', color='orange', edgecolor='black')
axes[0].set_xlabel('SOH (%)')
axes[0].set_ylabel('Count')
axes[0].set_title('SOH Distribution: Train vs Test')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Capacity distribution
axes[1].hist(train_df['capacity'], bins=30, alpha=0.7, label='Train (B0005)', color='blue', edgecolor='black')
axes[1].hist(test_df['capacity'], bins=30, alpha=0.7, label='Test (B0006+B0007)', color='orange', edgecolor='black')
axes[1].set_xlabel('Capacity (Ah)')
axes[1].set_ylabel('Count')
axes[1].set_title('Capacity Distribution: Train vs Test')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Train/Test Distribution Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/phase2/figures/train_test_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: results/phase2/figures/train_test_distribution.png")

## 8. Save Harmonized Data

In [ ]:
# Save all data
harmonizer.save_harmonized_data()

# Also save train/test splits separately
train_df.to_csv(f'{harmonizer.output_dir}/train_features.csv', index=False)
test_df.to_csv(f'{harmonizer.output_dir}/test_features.csv', index=False)

print(f"\n✓ Train features saved: {harmonizer.output_dir}/train_features.csv")
print(f"✓ Test features saved: {harmonizer.output_dir}/test_features.csv")

## 9. Harmonization Summary

In [ ]:
harmonizer.print_summary()

## 10. Verify Feature Consistency

In [ ]:
# Check for NaN/Inf values
print("Checking for data issues...")

nan_counts = features_df.isna().sum()
nan_features = nan_counts[nan_counts > 0]

if len(nan_features) > 0:
    print(f"\n⚠️ Features with NaN values:")
    print(nan_features)
else:
    print("\n✓ No NaN values found")

# Check for infinite values
numeric_df = features_df.select_dtypes(include=[np.number])
inf_counts = np.isinf(numeric_df).sum()
inf_features = inf_counts[inf_counts > 0]

if len(inf_features) > 0:
    print(f"\n⚠️ Features with Inf values:")
    print(inf_features)
else:
    print("✓ No Inf values found")

print("\n✓ Feature consistency check complete")

## Summary

**Step 2 Completed:**

✓ All NASA batteries loaded (B0005, B0006, B0007)  
✓ Cycles resampled to common time base  
✓ Physics features extracted (ICA, DCIR, temperature, etc.)  
✓ Features normalized (z-score)  
✓ Train/test split created (B0005 train, B0006+B0007 test)  
✓ All data saved to disk  
✓ Feature consistency verified  

**Output Files:**
- `data/phase2/harmonized/harmonized_features_*.csv`
- `data/phase2/harmonized/train_features.csv`
- `data/phase2/harmonized/test_features.csv`
- `data/phase2/harmonized/normalization_params_*.pkl`
- `data/phase2/harmonized/harmonized_cycles_*.pkl`

**Next Step:** Step 3 - Baseline Models (LSTM, CNN-LSTM, Transformer)